In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

In [ ]:
from openai import OpenAI
openai = OpenAI()


In [ ]:
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr 
import json

In [ ]:
reader = PdfReader("Vedant_Shekhar_Resume_ATS.pdf")
whole_texts = ""
for page in reader.pages:
    texts = page.extract_text()
    if(texts):
        whole_texts += texts

In [ ]:
print(whole_texts)

In [ ]:
model_name = "gpt-4o-mini"

In [ ]:
system_prompt = f""" 

#Your Role:

ok so your vedant's personal assistant named "Vedant4857" who will help people who want to know about vedant who has access to my resume : {whole_texts}
form this you are supposed to answer questions form ,
undertsnad the question asked refer my resume understand teh similarity and context and then answer answer on basis of that. 
Dont go too big answer only neccessary and good amount.

#Rules:

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""



In [ ]:
display(Markdown(system_prompt))

In [ ]:
messages = [
    {"role":"system","content":system_prompt},
    {"role":"user","content":"List projects done by you"}
]

In [ ]:
response = openai.chat.completions.create(model=model_name,messages=messages)
answer = response.choices[0].message.content
display(Markdown(answer))

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)